# Introduction to Machine Learning Force Fields

## Uppsala University - UFF Multiscale Materials Modelling Workshop

This notebook provides an introduction to machine learning-based force fields for materials and molecular simulations.

## 1. Why ML Force Fields?

Machine Learning Force Fields (MLFFs) bridge the gap between:

| Method | Accuracy | Speed | System Size |
|--------|----------|-------|-------------|
| DFT | High | Slow | ~100-1000 atoms |
| Classical FF | Low | Fast | ~10⁶ atoms |
| **ML Force Fields** | **High** | **Fast** | **~10⁴-10⁵ atoms** |

Key advantages:
- Near-DFT accuracy at a fraction of the computational cost
- Can capture complex many-body interactions
- Transferable across different configurations

## 2. Key Concepts

### 2.1 Atomic Descriptors

MLFFs require representations of atomic environments that are:
- **Invariant** to translation, rotation, and permutation
- **Smooth** and differentiable
- **Complete** (uniquely identify environments)

Common descriptors include:
- Atom-centered symmetry functions (Behler-Parrinello)
- Smooth overlap of atomic positions (SOAP)
- Graph neural network representations

In [ ]:
# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.kernel_ridge import KernelRidge

print("Libraries imported successfully!")

## 3. Simple Symmetry Function Example

We'll demonstrate a basic radial symmetry function (G2):

$$G_2^i = \sum_{j \neq i} e^{-\eta(R_{ij} - R_s)^2} f_c(R_{ij})$$

Where $f_c$ is a cutoff function.

In [ ]:
def cutoff_function(r, r_cut):
    """Cosine cutoff function."""
    return np.where(r < r_cut, 0.5 * (np.cos(np.pi * r / r_cut) + 1), 0.0)

def radial_symmetry_function(r, eta, Rs, r_cut):
    """Radial symmetry function G2."""
    return np.exp(-eta * (r - Rs)**2) * cutoff_function(r, r_cut)

# Visualize symmetry functions with different parameters
r = np.linspace(0, 6, 200)
r_cut = 5.0

plt.figure(figsize=(10, 5))

# Different eta values
plt.subplot(1, 2, 1)
for eta in [0.5, 1.0, 2.0, 4.0]:
    G2 = radial_symmetry_function(r, eta, Rs=0, r_cut=r_cut)
    plt.plot(r, G2, label=f'η = {eta}')
plt.xlabel('Distance (Å)', fontsize=12)
plt.ylabel('G2', fontsize=12)
plt.title('Varying η (Rs = 0)', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)

# Different Rs values
plt.subplot(1, 2, 2)
for Rs in [0, 1, 2, 3]:
    G2 = radial_symmetry_function(r, eta=2.0, Rs=Rs, r_cut=r_cut)
    plt.plot(r, G2, label=f'Rs = {Rs}')
plt.xlabel('Distance (Å)', fontsize=12)
plt.ylabel('G2', fontsize=12)
plt.title('Varying Rs (η = 2.0)', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Training a Simple ML Potential

Let's demonstrate the concept with a toy example using Kernel Ridge Regression.

In [ ]:
# Generate synthetic training data
np.random.seed(42)

# Simulate atomic distances as features
n_samples = 100
n_features = 10
X = np.random.uniform(1.0, 5.0, (n_samples, n_features))

# Generate "true" energies (synthetic DFT-like data)
def true_energy(X):
    """Synthetic potential energy surface."""
    E = np.zeros(len(X))
    for i, x in enumerate(X):
        E[i] = np.sum(np.exp(-0.5 * x) - 0.3 * np.exp(-0.2 * x**2))
    return E

y = true_energy(X) + np.random.normal(0, 0.01, n_samples)  # Add noise

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

In [ ]:
# Train Kernel Ridge Regression model
krr = KernelRidge(alpha=1e-3, kernel='rbf', gamma=0.1)
krr.fit(X_train, y_train)

# Predict
y_pred_train = krr.predict(X_train)
y_pred_test = krr.predict(X_test)

# Calculate errors
train_rmse = np.sqrt(np.mean((y_train - y_pred_train)**2))
test_rmse = np.sqrt(np.mean((y_test - y_pred_test)**2))

print(f"Training RMSE: {train_rmse:.4f} eV")
print(f"Test RMSE: {test_rmse:.4f} eV")

In [ ]:
# Visualize results
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.scatter(y_train, y_pred_train, alpha=0.7, label='Training')
plt.scatter(y_test, y_pred_test, alpha=0.7, label='Test')
lims = [min(y.min(), y_pred_test.min()), max(y.max(), y_pred_test.max())]
plt.plot(lims, lims, 'k--', alpha=0.5)
plt.xlabel('True Energy (eV)', fontsize=12)
plt.ylabel('Predicted Energy (eV)', fontsize=12)
plt.title('Parity Plot', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
errors = y_test - y_pred_test
plt.hist(errors, bins=15, edgecolor='black', alpha=0.7)
plt.xlabel('Error (eV)', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.title('Error Distribution (Test Set)', fontsize=14)
plt.axvline(x=0, color='r', linestyle='--', alpha=0.5)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Popular ML Force Field Methods

| Method | Architecture | Key Features |
|--------|-------------|---------------|
| Behler-Parrinello NN | Neural Network | Symmetry functions, atomic energies |
| GAP/SOAP | Gaussian Process | High accuracy, computationally intensive |
| SchNet | Graph Neural Network | Message passing, continuous filters |
| NequIP | Equivariant NN | E(3) equivariance, high data efficiency |
| MACE | Equivariant NN | Multi-body descriptors, state-of-the-art |

## 6. Summary

In this notebook, we covered:

- Motivation for ML force fields
- Key concepts: atomic descriptors and invariances
- Simple example of training an ML potential
- Overview of popular methods

### Next Steps

- Implement more sophisticated descriptors
- Train on real DFT data
- Run molecular dynamics with trained potentials

## References

1. Behler, J., & Parrinello, M. "Generalized neural-network representation of high-dimensional potential-energy surfaces." Phys. Rev. Lett. 98, 146401 (2007)
2. Bartók, A. P., et al. "Gaussian approximation potentials: The accuracy of quantum mechanics, without the electrons." Phys. Rev. Lett. 104, 136403 (2010)
3. Batzner, S., et al. "E(3)-equivariant graph neural networks for data-efficient and accurate interatomic potentials." Nat. Commun. 13, 2453 (2022)